# ✍️ Option 4 — Content Generator
## AI Engineer Ready · Cohort 1 · Assignment 1

---

## 📋 README — Read This First!

### 🎯 What This Assignment Is About
You will build a **multi-step AI content pipeline** that:
1. Takes a **topic** as input
2. Generates a structured **outline** (using an LLM)
3. Expands that outline into **full blog-style content** (using an LLM)
4. Produces a short **summary** of the full content (using an LLM)

The core idea is **chained LLM calls** — each step feeds into the next. This is the foundation of real-world AI pipelines.

---

### 🗂️ Files in This Project
| File | Purpose |
|---|---|
| `Content_Generator.ipynb` | This notebook — your main workspace |
| `main.py` | The same logic, packaged as a CLI script |
| `requirements.txt` | Python dependencies (`openai`, `langchain`) |
| `README.md` | Short project description |

---

### 🛠️ Tools & Libraries You Will Use
- **LangChain** — framework for chaining LLM calls
- **PromptTemplate** — structured, reusable prompts
- **OpenAI API** (or any compatible LLM)
- **Python f-strings / string formatting**

---

### 📊 How You Will Be Evaluated
| Criteria | What the Evaluator Looks For | Marks |
|---|---|---|
| **Setup & Installation** | Environment runs without errors | 10% |
| **Outline Generation** | LLM returns a meaningful, structured outline | 20% |
| **Content Expansion** | Each outline point is expanded into coherent paragraphs | 25% |
| **Summarization** | Summary is concise and captures the content | 20% |
| **Prompt Quality** | Prompts are clear, specific, and use PromptTemplate | 15% |
| **Code Clarity** | Clean code, comments, no leftover debug prints | 10% |

---

### ✅ Definition of Done
- [ ] All cells run top-to-bottom without errors
- [ ] Output is visible in the notebook (not just in `main.py`)
- [ ] Each of the 3 functions (`generate_outline`, `generate_content`, `summarize`) uses a real LLM call via LangChain
- [ ] `PromptTemplate` is used (not raw f-strings inside the LLM call)
- [ ] You can change the `TOPIC` variable in Section 6 and the whole pipeline re-runs correctly

---

> 💡 **Tip:** Work section by section. Each section builds on the previous one. Don't skip ahead!

---
# Section 1 — Environment Setup & Imports

## 📖 What to Do Here
Before writing any logic, you need to install the required libraries and import them.

### Step-by-step:
1. Run the `pip install` cell to install dependencies from `requirements.txt`
2. Import all the modules you will need throughout this notebook
3. Set your **OpenAI API key** (use an environment variable — never hardcode it!)

### 🔑 Getting Your API Key
- Sign up at https://platform.openai.com
- Go to **API Keys** and create a new secret key
- Store it safely — you'll set it as an environment variable below

### ⚠️ Evaluation Note
If this section throws an import error, the rest of the notebook cannot run. Make sure this section is 100% green before moving on.

In [ ]:
# Install dependencies
# Run this cell once. After installation, you can comment it out.
%pip install -r requirements.txt -q

In [2]:
# Import required libraries

import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Load environment variables from .env
load_dotenv()

print("✅ Imports successful")

✅ Imports successful


In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv(override=True)

# Retrieve the API key
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if not OPENAI_API_KEY:
    print("⚠️ WARNING: OPENAI_API_KEY is not set. LLM calls will fail.")
else:
    print(f"✅ API key loaded (ends with: ...{OPENAI_API_KEY[-4:]})")


✅ API key loaded (ends with: ...H9YA)


---
# Section 2 — Initialise the LLM

## 📖 What to Do Here
Create a single shared LLM object that all three pipeline steps will reuse.

### Concepts to Understand
- **`ChatOpenAI`** is a LangChain wrapper around OpenAI's chat models (e.g. `gpt-4o-mini`)
- **`temperature`** controls creativity: `0.0` = deterministic, `1.0` = very creative. Start with `0.7`.
- **`model_name`** — use `"gpt-4o-mini"` (cheap & fast for testing)

### 🔍 What the Evaluator Checks
- A single `llm` object is created here and reused below (not re-created in every function)
- Temperature is explicitly set

In [4]:
# Initialize the LLM

from langchain_openai import ChatOpenAI
import os

# Verify that the API key is available
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. Please run the API key setup cell first."
    )

# Initialize the OpenAI chat model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7
)

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


---
# Section 3 — Step 1: Generate an Outline

## 📖 What to Do Here
Implement the `generate_outline(topic)` function. It should ask the LLM to produce a structured outline for a blog post on the given topic.

### How It Works
```
topic (string)  ──►  PromptTemplate  ──►  LLM  ──►  outline (list of strings)
```

### Step-by-step Instructions
1. **Create a `PromptTemplate`** with a variable `{topic}`. The prompt should instruct the LLM to return a numbered outline with 4-6 sections.
2. **Format the prompt** by calling `prompt.format(topic=topic)`
3. **Call the LLM** with the formatted prompt
4. **Parse the response** — split by newlines, strip empty lines, return a Python list

### 💡 Suggested Prompt (you can improve it!)
```
You are a professional content strategist.
Create a detailed blog post outline for the topic: "{topic}".
Return ONLY a numbered list of 5 section headings, one per line.
Do not include any explanation.
```

### 🔍 What the Evaluator Checks
- `PromptTemplate` is used (not a raw f-string)
- Function returns a Python `list`, not a raw string
- Output has at least 3 meaningful headings for any topic

In [5]:
from langchain_core.prompts import PromptTemplate

# Define the outline prompt template
outline_prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
You are an expert content writer and subject matter specialist.

Create a clear, well-structured outline for the following topic:

Topic: {topic}

Instructions:
1. Provide a suitable title.
2. Organize the outline into logical main sections.
3. Include relevant subsections under each main section.
4. Arrange the sections in a logical sequence.
5. Cover the most important aspects of the topic.
6. Return only the outline, using numbered headings and bullet points.
"""
)

print("✅ Outline prompt template created successfully!")


✅ Outline prompt template created successfully!


In [ ]:
def generate_outline(topic: str) -> list:
    """
    Uses an LLM to generate a structured outline for a blog post on `topic`.

    Args:
        topic (str): The subject of the blog post.

    Returns:
        list: A list of section heading strings.
    """
    # Step 1: Format the prompt with the given topic
    formatted_prompt = outline_prompt.format(topic=topic)

    # Step 2: Invoke the LLM
    response = llm.invoke(formatted_prompt)

    # Step 3: Extract the generated text
    text = response.content

    # Step 4: Split the text into individual lines
    lines = text.split("\n")

    # Step 5: Strip whitespace and remove empty lines
    outline = [line.strip() for line in lines if line.strip()]

    # Step 6: Return the list of outline items
    return outline


# Quick test
test_outline = generate_outline("Getting started with LangChain")

print("Generated Outline:")
for i, point in enumerate(test_outline, 1):
    print(f"  {i}. {point}")

Generated Outline:
  1. # Getting Started with LangChain: A Comprehensive Guide
  2. ## 1. Introduction to LangChain
  3. - 1.1 What is LangChain?
  4. - 1.2 Purpose and Applications
  5. - 1.3 Importance in Natural Language Processing
  6. ## 2. Key Concepts of LangChain
  7. - 2.1 Chain of Thought
  8. - 2.1.1 Definition and Explanation
  9. - 2.1.2 Use Cases
  10. - 2.2 Language Models
  11. - 2.2.1 Overview of Language Models
  12. - 2.2.2 Integration with LangChain
  13. - 2.3 Agents and Tools
  14. - 2.3.1 Understanding Agents
  15. - 2.3.2 Common Tools in LangChain
  16. ## 3. Setting Up the Environment
  17. - 3.1 System Requirements
  18. - 3.2 Installation Guide
  19. - 3.2.1 Using pip
  20. - 3.2.2 Alternative Installation Methods
  21. - 3.3 Basic Configuration
  22. - 3.3.1 Setting Up API Keys
  23. - 3.3.2 Environment Variables
  24. ## 4. Building Your First LangChain Application
  25. - 4.1 Creating a Simple Chain
  26. - 4.1.1 Step-by-Step Example
  27. - 4.1.2 Explana

---
# Section 4 — Step 2: Generate Full Content

## 📖 What to Do Here
Implement `generate_content(outline)`. It takes the list of headings from Step 1 and asks the LLM to write a full paragraph for each section.

### How It Works
```
outline (list)  ──►  PromptTemplate  ──►  LLM  ──►  full_content (string)
```

### Step-by-step Instructions
1. **Join the outline list** into a single string (one heading per line) so it can be injected into the prompt
2. **Create a `PromptTemplate`** with variable `{outline}`. The prompt should ask the LLM to write 2-3 paragraphs per section.
3. **Format & invoke** the LLM
4. Return the full content as a single string

### 💡 Suggested Prompt
```
You are an expert blog writer.
Using the following outline, write a complete blog post.
For each section heading, write 2-3 informative paragraphs.
Use markdown formatting (## for headings).

Outline:
{outline}
```

### 🔍 What the Evaluator Checks
- Input is the outline list from Step 1 (not hardcoded)
- Each outline heading appears in the output as a section
- Content is coherent and meaningful (not placeholder text)
- Returns a string, not a list

In [9]:
from langchain_core.prompts import PromptTemplate

# Define the content prompt template
content_prompt = PromptTemplate(
    input_variables=["topic", "outline"],
    template="""
You are an expert content writer and subject matter specialist.

Write a detailed, informative, and well-structured blog post using the following information.

Topic: {topic}

Outline:
{outline}

Instructions:
1. Start with an engaging title.
2. Write a compelling introduction explaining the topic.
3. Follow the provided outline and cover each section in order.
4. Explain concepts clearly using simple, easy-to-understand language.
5. Include practical examples wherever appropriate.
6. Use headings and subheadings to organize the content.
7. Ensure smooth transitions between sections.
8. End with a clear conclusion summarizing the key takeaways.
9. Avoid unnecessary repetition and unsupported claims.
10. Return only the completed blog post.
"""
)

print("✅ Content prompt template created successfully!")

✅ Content prompt template created successfully!


In [10]:
def generate_content(outline: list) -> str:
    """
    Expands an outline into a full blog post using an LLM.

    Args:
        outline (list): List of section headings from generate_outline().

    Returns:
        str: The full blog post content as a markdown-formatted string.
    """
    # Step 1: Join the outline list into a single string
    outline_str = "\n".join(outline)

    # Step 2: Format the content prompt with the outline
    formatted_prompt = content_prompt.format(
        topic="Getting started with LangChain",
        outline=outline_str
    )

    # Step 3: Invoke the LLM
    response = llm.invoke(formatted_prompt)

    # Step 4: Return the generated content
    return response.content


# Quick test
test_content = generate_content(
    test_outline or ["Introduction", "Key Concepts", "Conclusion"]
)

print("Generated Content (first 500 chars):")
print((test_content or "")[:500], "...")

Generated Content (first 500 chars):
# Getting Started with LangChain: A Comprehensive Guide

## 1. Introduction to LangChain

### 1.1 What is LangChain?
LangChain is an innovative framework designed to streamline the development of applications powered by large language models (LLMs). It provides a modular approach, allowing developers to create complex workflows that can intelligently process and manipulate language data. By facilitating the integration of various components such as language models, APIs, and tools, LangChain ena ...


---
# Section 5 — Step 3: Summarize the Content

## 📖 What to Do Here
Implement `summarize(content)`. It takes the full blog post and produces a concise 3-5 sentence summary.

### How It Works
```
full_content (string)  ──►  PromptTemplate  ──►  LLM  ──►  summary (string)
```

### Step-by-step Instructions
1. **Create a `PromptTemplate`** with variable `{content}`.
2. **Be careful with long content** — you can truncate the input to the first 3000 characters if needed to stay within token limits.
3. **Format & invoke** the LLM
4. Return the summary string

### 💡 Suggested Prompt
```
You are a content editor. Summarize the following blog post in 3-5 sentences.
Be concise and capture the key points.

Blog Post:
{content}

Summary:
```

### 🔍 What the Evaluator Checks
- Summary is 3-5 sentences (not a one-liner, not a full rewrite)
- Summary accurately reflects the content (not generic filler)
- Input comes from Step 2's output (not hardcoded)

In [11]:
from langchain_core.prompts import PromptTemplate

# Define the summarization prompt template
summary_prompt = PromptTemplate(
    input_variables=["content"],
    template="""
You are an expert summarization assistant.

Summarize the following blog post while preserving its most important information.

BLOG POST:
{content}

Instructions:
1. Identify the main topic and central message.
2. Highlight the most important concepts and key takeaways.
3. Preserve essential facts, explanations, and examples.
4. Remove redundant information and unnecessary details.
5. Use clear, concise, and easy-to-understand language.
6. Organize the summary using headings and bullet points where appropriate.
7. Do not introduce information that is not present in the original content.
8. Keep the summary significantly shorter than the original blog post.
9. Return only the summary.
"""
)

print("✅ Summarization prompt template created successfully!")


✅ Summarization prompt template created successfully!


In [12]:
def summarize(content: str) -> str:
    """
    Produces a 3-5 sentence summary of a blog post using an LLM.

    Args:
        content (str): The full blog post text from generate_content().

    Returns:
        str: A short summary paragraph.
    """
    # Step 1: Truncate content to avoid token limits
    content_trimmed = content[:3000]

    # Step 2: Format the summarization prompt
    formatted_prompt = summary_prompt.format(
        content=content_trimmed
    )

    # Step 3: Invoke the LLM
    response = llm.invoke(formatted_prompt)

    # Step 4: Return the generated summary
    return response.content


# Quick test
test_summary = summarize(
    test_content or "Sample content for testing."
)

print("Summary:")
print(test_summary)

Summary:
# Summary of Getting Started with LangChain

## Introduction to LangChain
- **What is LangChain?**
  - A framework designed to simplify the development of applications using large language models (LLMs).
  - Supports modular workflows for processing and manipulating language data.

- **Purpose and Applications**
  - Empowers developers to create diverse applications like chatbots, content generation tools, and data extraction systems.
  - Capable of performing tasks such as summarization, translation, and code generation.

- **Importance in NLP**
  - Provides essential tools for harnessing the power of language models, making NLP application development accessible to more developers.

## Key Concepts of LangChain
- **Chain of Thought**
  - Refers to the logical sequence of reasoning steps a language model takes to reach conclusions.
  - Useful for multi-step reasoning tasks, enhancing human-like reasoning in applications.

- **Language Models**
  - Algorithms that understand a

---
# Section 6 — 🚀 Run the Full Pipeline

## 📖 What to Do Here
Now that all three functions are implemented, wire them together into a single end-to-end pipeline.

### Instructions
1. Set your `TOPIC` below (change it to anything you like!)
2. Call `generate_outline` → `generate_content` → `summarize` in sequence
3. Print each result with a clear label

### 🔍 What the Evaluator Checks
- Changing `TOPIC` and re-running produces different, topic-appropriate output
- All three outputs are printed and visible in the notebook
- No hardcoded strings — everything flows from the `TOPIC` variable

In [13]:
# Configuration — change this to test your pipeline!
TOPIC = "Getting started with LangChain"

print(f"🔄 Running Content Generator Pipeline for topic: '{TOPIC}'")
print("=" * 60)

# Step 1: Generate Outline
print("\n📝 STEP 1 — Generating Outline...")
outline = generate_outline(TOPIC)

if outline:
    print("Outline:")
    for i, point in enumerate(outline, 1):
        print(f"  {i}. {point}")
else:
    print("⚠️ Outline not generated.")

# Step 2: Generate Content
print("\n📄 STEP 2 — Generating Full Content...")

# Pass the outline and current topic to generate_content()
content = generate_content(outline, topic=TOPIC) if outline else None

if content:
    print("Content (first 800 chars):")
    print(content[:800], "...")
else:
    print("⚠️ Content not generated.")

# Step 3: Summarize
print("\n🔍 STEP 3 — Summarizing...")
summary = summarize(content) if content else None

if summary:
    print("Summary:")
    print(summary)
else:
    print("⚠️ Summary not generated.")

print("\n" + "=" * 60)
print("✅ Pipeline complete!")

🔄 Running Content Generator Pipeline for topic: 'Getting started with LangChain'

📝 STEP 1 — Generating Outline...
Outline:
  1. # Getting Started with LangChain: A Comprehensive Guide
  2. ## 1. Introduction to LangChain
  3. - 1.1 What is LangChain?
  4. - 1.2 Importance and Applications
  5. - 1.2.1 Use Cases in Natural Language Processing
  6. - 1.2.2 Integration with Other Technologies
  7. - 1.3 Overview of Key Features
  8. ## 2. Setting Up the Environment
  9. - 2.1 System Requirements
  10. - 2.2 Installation Guide
  11. - 2.2.1 Installing via pip
  12. - 2.2.2 Alternative Installation Methods
  13. - 2.3 Setting Up a Development Environment
  14. - 2.3.1 Recommended IDEs
  15. - 2.3.2 Configuring Virtual Environments
  16. ## 3. Core Concepts of LangChain
  17. - 3.1 Chains
  18. - 3.1.1 What are Chains?
  19. - 3.1.2 Types of Chains
  20. - 3.2 Agents
  21. - 3.2.1 Introduction to Agents
  22. - 3.2.2 How Agents Operate
  23. - 3.3 Memory
  24. - 3.3.1 Understanding Memory i

TypeError: generate_content() got an unexpected keyword argument 'topic'

---
# Section 7 — Self-Assessment Checklist

Before submitting, go through this checklist. Be honest with yourself!

Run the cell below and update the `checklist` dictionary — set each item to `True` when you've completed it.

In [18]:
# Configuration
TOPIC = "Getting started with LangChain"

# Step 1: Generate outline
print("📝 Generating outline...")
outline = generate_outline(TOPIC)

# Step 2: Generate content
print("📄 Generating blog content...")
content = generate_content(outline, topic=TOPIC) if outline else ""

# Step 3: Generate summary
print("🔍 Generating summary...")
summary = summarize(content) if content else ""

# Display results
print("\n✅ Pipeline Results")
print("=" * 50)

print("\nOUTLINE:")
print("\n".join(outline) if outline else "No outline generated.")

print("\nCONTENT (first 500 characters):")
print(content[:500] if content else "No content generated.")

print("\nSUMMARY:")
print(summary if summary else "No summary generated.")

📝 Generating outline...
📄 Generating blog content...


TypeError: generate_content() got an unexpected keyword argument 'topic'

In [17]:
# Self-Assessment Checklist
checklist = {
    "All cells run without errors": False,
    "API key loaded via environment variable": False,
    "LLM object initialised with explicit temperature": False,
    "PromptTemplate used in all 3 functions": False,
    "generate_outline returns a list": False,
    "generate_content returns a string with sections": False,
    "summarize returns a 3-5 sentence summary": False,
    "Pipeline runs end-to-end from TOPIC variable": False,
    "Changing TOPIC produces correct new output": False,
    "Code is clean and commented": False,
}

# 1. Check API key
api_key_loaded = bool(os.environ.get("OPENAI_API_KEY"))
checklist["API key loaded via environment variable"] = api_key_loaded

# 2. Check LLM initialization and temperature
try:
    checklist["LLM object initialised with explicit temperature"] = (
        llm is not None and llm.temperature == 0.7
    )
except (NameError, AttributeError):
    checklist["LLM object initialised with explicit temperature"] = False

# 3. Check prompt templates
try:
    checklist["PromptTemplate used in all 3 functions"] = all(
        isinstance(prompt, PromptTemplate)
        for prompt in [outline_prompt, content_prompt, summary_prompt]
    )
except NameError:
    checklist["PromptTemplate used in all 3 functions"] = False

# 4. Check outline output
try:
    checklist["generate_outline returns a list"] = (
        isinstance(outline, list) and len(outline) > 0
    )
except NameError:
    checklist["generate_outline returns a list"] = False

# 5. Check generated content
try:
    checklist["generate_content returns a string with sections"] = (
        isinstance(content, str)
        and len(content.strip()) > 0
        and len(content.strip().splitlines()) > 1
    )
except NameError:
    checklist["generate_content returns a string with sections"] = False

# 6. Check summary length
try:
    import re

    sentences = [
        s for s in re.split(r'(?<=[.!?])\s+', summary.strip())
        if s.strip()
    ]
    checklist["summarize returns a 3-5 sentence summary"] = (
        isinstance(summary, str) and 3 <= len(sentences) <= 5
    )
except (NameError, AttributeError):
    checklist["summarize returns a 3-5 sentence summary"] = False

# 7. Check end-to-end pipeline
checklist["Pipeline runs end-to-end from TOPIC variable"] = all([
    isinstance(TOPIC, str) and bool(TOPIC.strip()),
    isinstance(outline, list) and len(outline) > 0,
    isinstance(content, str) and bool(content.strip()),
    isinstance(summary, str) and bool(summary.strip()),
])

# These two items require manual verification
# Mark True only after checking the relevant behavior.
checklist["All cells run without errors"] = False  # Update after checking notebook
checklist["Changing TOPIC produces correct new output"] = False  # Test another topic

NameError: name 'content' is not defined